# Exp 38: Ensemble plateau thresholds

Card: [exp38_ensemble_plateau_thresholds.md](../../docs/experiments/exp38_ensemble_plateau_thresholds.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_curve

from core.training.config import CLASSES, ROOT
from core.training.runs import PROB_COLS
from core.analysis.thresholds import best_f1, cross_fold_f1, plateau_f1
from core.training.ensemble import Ensemble
from core.analysis.plots import save_figures, use_style

CONFIG_DIR = ROOT / "configs" / "models"
MODELS = {
    "exp26 ResNet-18": CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml",
    "EfficientNet-B0": CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.yaml",
    "RegNetY-016": CONFIG_DIR / "exp33_regnety_016" / "regnety_016.yaml",
    "ConvNeXt-Tiny": CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.yaml",
}
MARGIN = 0.05  # plateau: F1 >= max - MARGIN
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp38_ensemble_plateau_thresholds"

# dataviz reference palette, light mode
INK, PEAK, MIDDLE = "#52514e", "#eb6834", "#2a78d6"
use_style()

## Ensemble


In [ ]:
oofs = Ensemble.member_oofs(MODELS.values())
# mean of out-of-fold probs per class; labels and folds shared by all members
oof = oofs[0][["fold", *CLASSES]].join(sum(o[PROB_COLS] for o in oofs) / len(oofs))
labels, probs = oof[CLASSES].to_numpy(), oof[PROB_COLS].to_numpy()

## F1 against threshold


In [ ]:
peak, plateau = best_f1(labels, probs, CLASSES), plateau_f1(labels, probs, CLASSES, MARGIN)
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.6 * len(CLASSES), 3.6), sharey=True
)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
    f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
    ax.axvspan(
        plateau.loc[cls, "low"],
        plateau.loc[cls, "high"],
        color=MIDDLE,
        alpha=0.12,
        lw=0,
        label=f"F1 within {MARGIN} of max",
    )
    ax.plot(thr, f1, color=INK, lw=1.5)
    best_t, mean_t = peak.loc[cls, "threshold"], plateau.loc[cls, "threshold"]
    ax.axvline(best_t, color=PEAK, lw=2, ls="--", label="best_t (peak)")
    ax.axvline(mean_t, color=MIDDLE, lw=2, label="mean_t (plateau middle)")
    # value labels at two heights, outward from the pair so they never cross a line; both left near right edge
    for t, label, y in [(best_t, "best_t", 0.22), (mean_t, "mean_t", 0.08)]:
        right = t == max(best_t, mean_t) and t < 0.8
        ax.text(
            t + 0.015 if right else min(best_t, mean_t) - 0.015,
            y,
            f"{label} {t:.3f}",
            ha="left" if right else "right",
            fontsize=8,
            color=INK,
        )
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("ensemble\nF1")
fig.suptitle(
    f"F1 against threshold, mean of {len(MODELS)} models, out-of-fold rows of folds 0–4",
    color=INK,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.93),
)
figures = {f"{PREFIX}_f1_curves.png": fig}
plt.show()

## Peak vs plateau thresholds


In [ ]:
RULES = {"peak": None, "plateau": MARGIN}  # cross_fold_f1 margin: None = best F1

table = pd.concat(
    {
        "peak": best_f1(labels, probs, CLASSES)[["threshold", "f1"]],
        "plateau": plateau_f1(labels, probs, CLASSES, MARGIN)[["threshold", "f1"]],
    },
    axis=1,
)
honest = pd.Series(
    {rule: cross_fold_f1(labels, probs, oof["fold"], CLASSES, margin).mean() for rule, margin in RULES.items()},
    name="honest Macro F1",
)
display(table.round(3), honest.round(3))

In [ ]:
save_figures(figures, FIGURES)